In [9]:
import sys
import requests
import zipfile
import os
import random
train_path_in = "data/test_data/imdb/train.txt"
train_path_out = "data/test_data/imdb/train_subset.txt"
test_path_in = "data/test_data/imdb/test.txt"
test_path_out = "data/test_data/imdb/test_subset.txt"

data_path = "data/test_data/imdb/"

random_seed = 42 # Random seed for experiments
random.seed(random_seed)

In [2]:
session = requests.Session()
response = session.get("https://drive.google.com/a/illinois.edu/uc?export=download&id=1c8X_Ooth2fQleCVz2gCXlOd3-zzE9Mws", stream=True)

In [3]:
path = "data/test_data/imdb.zip"

if not os.path.exists("data/test_data"):
    os.mkdir("data/test_data")

CHUNK_SIZE = 32768
with open(path, "wb") as f:
    for chunk in response.iter_content(CHUNK_SIZE):
        if chunk:  # filter out keep-alive new chunks
            f.write(chunk)

In [4]:
if not os.path.exists("data/test_data/imdb"):
    os.mkdir("data/test_data/imdb")
with zipfile.ZipFile(path, 'r') as z: z.extractall("data/test_data/imdb")

In [5]:
def subset_data(in_path, out_path, percentage):
    with open(in_path, 'r', encoding='utf-8') as file:
        all_lines = file.readlines()
    label_file = in_path.replace(".txt", "_labels.txt")
    with open(label_file, 'r', encoding='utf-8') as file:
        all_labels = file.readlines()

    pairs = []
    for i in range(len(all_lines)):
        pairs.append((all_lines[i], all_labels[i]))
    
    
    sample_size = int(len(all_lines) * percentage)
    sampled_lines = random.sample(pairs, sample_size)
    with open(out_path, 'w', encoding='utf-8') as file:
        file.writelines([pair[0] for pair in sampled_lines])
    label_out_path = out_path.replace(".txt", "_labels.txt")
    with open(label_out_path, 'w', encoding='utf-8') as file:
        file.writelines([pair[1] for pair in sampled_lines])
    print("Original number of lines: ", len(all_lines))
    print("New number of lines: ", len(sampled_lines))

def rename_subset_data(data_path):
    if os.path.isfile(data_path + "train_subset.txt"):
        os.rename(data_path + "train.txt", data_path + "train_og.txt")
        os.rename(data_path + "train_subset.txt", data_path + "train.txt")
        os.rename(data_path + "train_labels.txt", data_path + "train_labels_og.txt")
        os.rename(data_path + "train_subset_labels.txt", data_path + "train_labels.txt")

    if os.path.isfile(data_path + "test_subset.txt"):
        os.rename(data_path + "test.txt", data_path + "test_og.txt")
        os.rename(data_path + "test_subset.txt", data_path + "test.txt")
        os.rename(data_path + "test_labels.txt", data_path + "test_labels_og.txt")
        os.rename(data_path + "test_subset_labels.txt", data_path + "test_labels.txt")

In [6]:
#subset_data(train_path_in, train_path_out, 0.5)
#subset_data(test_path_in, test_path_out, 0.5)
#rename_subset_data(data_path)

In [3]:
import sys
sys.path.append('KeyClass-single-label/keyclass/')
sys.path.append('KeyClass-single-label/scripts/')


import argparse
import label_data, encode_datasets, train_downstream_model
import torch
import pickle
import numpy as np
import os
from os.path import join, exists
from datetime import datetime
import utils
import models
import create_lfs
import train_classifier
import importlib

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Jon\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [5]:
# Input arguments
config_file_path = r'test_config.yml' # Specify path to the configuration file
default_config_path = r'KeyClass-single-label/config_files/default_config.yml'

In [9]:
importlib.reload(utils)

args = utils.Parser(config_file_path=config_file_path, default_config_file_path=default_config_path).parse()

if args['use_custom_encoder']:
    model = models.CustomEncoder(pretrained_model_name_or_path=args['base_encoder'], 
        device='cuda' if torch.cuda.is_available() else 'cpu')
else:
    model = models.Encoder(model_name=args['base_encoder'], 
        device='cuda' if torch.cuda.is_available() else 'cpu')

for split in ['train', 'test']:
    sentences = utils.fetch_data(dataset=args['dataset'], split=split, path=args['data_path'])
    embeddings = model.encode(sentences=sentences, batch_size=args['end_model_batch_size'], 
                                show_progress_bar=args['show_progress_bar'], 
                                normalize_embeddings=args['normalize_embeddings'])
    with open(join(args['data_path'], args['dataset'], f'{split}_embeddings.pkl'), 'wb') as f:
        pickle.dump(embeddings, f)

INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: paraphrase-mpnet-base-v2


Batches:   0%|          | 0/196 [00:00<?, ?it/s]

Batches:   0%|          | 0/196 [00:00<?, ?it/s]

In [10]:
# Load training data
train_text = utils.fetch_data(dataset=args['dataset'], path=args['data_path'], split='train')

training_labels_present = False
if exists(join(args['data_path'], args['dataset'], 'train_labels.txt')):
    with open(join(args['data_path'], args['dataset'], 'train_labels.txt'), 'r') as f:
        y_train = f.readlines()
    y_train = np.array([int(i.replace('\n','')) for i in y_train])
    training_labels_present = True
else:
    y_train = None
    training_labels_present = False
    print('No training labels found!')

with open(join(args['data_path'], args['dataset'], 'train_embeddings.pkl'), 'rb') as f:
    X_train = pickle.load(f)

# Print dataset statistics
print(f"Getting labels for the {args['dataset']} data...")
print(f'Size of the data: {len(train_text)}')
if training_labels_present:
    print('Class distribution', np.unique(y_train, return_counts=True))

# Load label names/descriptions
label_names = []
for a in args:
    if 'target' in a: label_names.append(args[a])

# Creating labeling functions
labeler = create_lfs.CreateLabellingFunctions(base_encoder=args['base_encoder'], 
                                            device=torch.device(args['device']),
                                            label_model=args['label_model'])
proba_preds = labeler.get_labels(text_corpus=train_text, label_names=label_names, min_df=args['min_df'], 
                                ngram_range=args['ngram_range'], topk=args['topk'], y_train=y_train, 
                                label_model_lr=args['label_model_lr'], label_model_n_epochs=args['label_model_n_epochs'], 
                                verbose=True, n_classes=args['n_classes'])

y_train_pred = np.argmax(proba_preds, axis=1)

# Save the predictions
if not os.path.exists(args['preds_path']): os.makedirs(args['preds_path'])
with open(join(args['preds_path'], f"{args['label_model']}_proba_preds.pkl"), 'wb') as f:
    pickle.dump(proba_preds, f)

# Print statistics
print('Label Model Predictions: Unique value and counts', np.unique(y_train_pred, return_counts=True))
if training_labels_present:
    print('Label Model Training Accuracy', np.mean(y_train_pred==y_train))

INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: paraphrase-mpnet-base-v2


Getting labels for the imdb data...
Size of the data: 25000
Class distribution (array([0, 1]), array([12500, 12500], dtype=int64))
Found assigned category counts [6786 9581]
labeler.vocabulary:
 16367
labeler.word_indicator_matrix.shape (25000, 600)
Len keywords 600
assigned_category: Unique and Counts (array([0, 1], dtype=int64), array([300, 300], dtype=int64))
negative, hate, expensive, bad, poor, broke, waste, horrible, would not recommend ['abominable' 'abomination' 'absolute worst' 'absolutely awful'
 'absolutely terrible' 'abuse' 'abused' 'abusive' 'abysmal'
 'acting horrible' 'acting poor' 'acting terrible' 'actors bad'
 'actually bad' 'also bad' 'among worst' 'annoyance' 'annoying' 'appalled'
 'appalling' 'atrocious' 'awful' 'awfully' 'awfulness' 'bad' 'bad actor'
 'bad actors' 'bad actually' 'bad almost' 'bad bad' 'bad could'
 'bad either' 'bad enough' 'bad even' 'bad film' 'bad films' 'bad get'
 'bad horror' 'bad idea' 'bad like' 'bad made' 'bad makes' 'bad many'
 'bad movie'

INFO:root:Computing O...
INFO:root:Estimating \mu...
INFO:root:Using GPU...
100%|█████████████████████████████████████████████████████████████████████████████| 100/100 [00:03<00:00, 28.12epoch/s]
INFO:root:Finished Training


Label Model Predictions: Unique value and counts (array([0, 1], dtype=int64), array([ 8910, 16090], dtype=int64))
Label Model Training Accuracy 0.69992


In [6]:
importlib.reload(utils)

args = utils.Parser(config_file_path=config_file_path, default_config_file_path=default_config_path).parse()

# Set random seeds
random_seed = random_seed
torch.manual_seed(random_seed)
np.random.seed(random_seed)

X_train_embed_masked, y_train_lm_masked, y_train_masked, \
	X_test_embed, y_test, training_labels_present, \
	sample_weights_masked, proba_preds_masked = train_downstream_model.load_data(args)

# Train a downstream classifier

if args['use_custom_encoder']:
	encoder = models.CustomEncoder(pretrained_model_name_or_path=args['base_encoder'], device=args['device'])
else:
	encoder = models.Encoder(model_name=args['base_encoder'], device=args['device'])

classifier = models.FeedForwardFlexible(encoder_model=encoder,
										h_sizes=args['h_sizes'], 
										activation=eval(args['activation']),
										device=torch.device(args['device']))
print('\n===== Training the downstream classifier =====\n')
model = train_classifier.train(model=classifier, 
							device=torch.device(args['device']),
							X_train=X_train_embed_masked, 
							y_train=y_train_lm_masked,
							sample_weights=sample_weights_masked if args['use_noise_aware_loss'] else None, 
							epochs=args['end_model_epochs'], 
							batch_size=args['end_model_batch_size'], 
							criterion=eval(args['criterion']), 
							raw_text=False, 
							lr=eval(args['end_model_lr']), 
							weight_decay=eval(args['end_model_weight_decay']),
							patience=args['end_model_patience'])


end_model_preds_train = model.predict_proba(torch.from_numpy(X_train_embed_masked), batch_size=512, raw_text=False)
end_model_preds_test = model.predict_proba(torch.from_numpy(X_test_embed), batch_size=512, raw_text=False)

INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: paraphrase-mpnet-base-v2


Confidence of least confident data point of class 0: 0.911190136272153
Confidence of least confident data point of class 1: 0.9999166207780085

==== Data statistics ====
Size of training data: (25000, 768), testing data: (25000, 768)
Size of testing labels: (25000,)
Size of training labels: (25000,)
Training class distribution (ground truth): [0.5 0.5]
Training class distribution (label model predictions): [0.3564 0.6436]

KeyClass only trains on the most confidently labeled data points! Applying mask...

==== Data statistics (after applying mask) ====
Size of training data: (7000, 768)
Size of training labels: (7000,)
Training class distribution (ground truth): [0.55042857 0.44957143]
Training class distribution (label model predictions): [0.5 0.5]

===== Training the downstream classifier =====



Epoch 19:  95%|████████████▎| 19/20 [00:03<00:00,  6.09batch/s, best_loss=0.546, running_loss=0.551, tolerance_count=3]

Stopping early...


In [7]:
y_test

array([1, 1, 1, ..., 0, 0, 0])

In [8]:
args = utils.Parser(config_file_path=config_file_path, default_config_file_path=default_config_path).parse()

print('\n===== Self-training the downstream classifier =====\n')
# Fetching the raw text data for self-training
X_train_text = utils.fetch_data(dataset=args['dataset'], path=args['data_path'], split='train')
X_test_text = utils.fetch_data(dataset=args['dataset'], path=args['data_path'], split='test')

model = train_classifier.self_train(model=model, 
									X_train=X_train_text, 
									X_val=X_test_text, 
									y_val=y_test, 
									device=torch.device(args['device']), 
									lr=eval(args['self_train_lr']), 
									weight_decay=eval(args['self_train_weight_decay']),
									patience=args['self_train_patience'], 
									batch_size=args['self_train_batch_size'], 
									q_update_interval=args['q_update_interval'],
									self_train_thresh=eval(args['self_train_thresh']), 
									print_eval=True)

# save the model
current_time = datetime.now()
model_name = f'end_model_self_trained_{current_time.strftime("%d-%b-%Y-%H_%M_%S")}.pth'
print(f'Saving model {model_name}...')
with open(join(args['model_path'], model_name), 'wb') as f:
    torch.save(model, f)

end_model_preds_test = model.predict_proba(X_test_text, batch_size=args['self_train_batch_size'], raw_text=True)
# Save the predictions
with open(
        join(args['preds_path'], 'end_model_self_trained_preds_test.pkl'),
        'wb') as f:
    pickle.dump(end_model_preds_test, f)

# Print statistics
testing_metrics = utils.compute_metrics_bootstrap(y_preds=np.argmax(end_model_preds_test, axis=1),
													y_true=y_test, 
													average=args['average'], 
													n_bootstrap=args['n_bootstrap'], 
													n_jobs=args['n_jobs'])
print(testing_metrics)


===== Self-training the downstream classifier =====



Epoch 10:   8%| | 10/125 [32:44<6:16:31, 196.45s/batch, self_train_agreement=1, tolerance_count=2, validation_accuracy=


Saving model end_model_self_trained_29-Apr-2025-03_18_54.pth...


[Parallel(n_jobs=10)]: Using backend LokyBackend with 10 concurrent workers.


[[0.9073444  0.00183523]
 [0.90766566 0.00182696]
 [0.9073444  0.00183523]
 [0.907505   0.0018305 ]]


[Parallel(n_jobs=10)]: Done  30 tasks      | elapsed:    2.1s
[Parallel(n_jobs=10)]: Done 100 out of 100 | elapsed:    2.2s finished
